In [27]:
from IPython.display import Image

- 解码-编码有很多情况不可逆
    - D(E(text)) = text
    - E(D(token_ids)) ≠ token_ids
        - 即先对 token ids（model generation）进行解码，再进行编码之后的 token ids 有可能不等同于原始的 model generated 的 ids
- encode: 贪心/最长匹配
    - 生成时：模型每步从整个词表里挑一个 token。它可能依次挑出 <、think、> 这条“非规范”路径（哪怕词表里也存在 <th、ink、> 这样的更长/更“规整”的片段）。
    - 比如生成”helloworld”的token可能有几种组合情况，但是根据”helloworld”转成的token只有一种组合，可能跟原来的token不同

In [26]:
Image(url='./figs/encode-decode.png', width=400)

In [8]:
from transformers import AutoTokenizer
T = AutoTokenizer.from_pretrained('Qwen/Qwen2.5-7B-Instruct', use_fast=True)

In [31]:
T.encode('<think>')

[13708, 766, 29]

In [23]:
# <th, ink, >
T.decode([13708, 766, 29]), T.encode(T.decode([13708, 766, 29]))

('<think>', [13708, 766, 29])

In [24]:
# <, think, >
T.decode([27, 26865, 29]), T.encode(T.decode([27, 26865, 29]))

('<think>', [13708, 766, 29])

In [36]:
[T.decode(token_id) for token_id in [13708, 766, 29]], [T.decode(token_id) for token_id in [27, 26865, 29]]

(['<th', 'ink', '>'], ['<', 'think', '>'])

In [12]:
ids_lt   = T.encode("<", add_special_tokens=False)
ids_mid  = T.encode("think", add_special_tokens=False)
ids_rt   = T.encode(">", add_special_tokens=False)
ids = ids_lt + ids_mid + ids_rt

In [17]:
ids, T.convert_ids_to_tokens(ids), T.decode(ids)

([27, 26865, 29], ['<', 'think', '>'], '<think>')

In [22]:
ids, T.decode(ids), T.encode(T.decode(ids)), T.decode(T.encode(T.decode(ids)))

([27, 26865, 29], '<think>', [13708, 766, 29], '<think>')

### training issues

In [30]:
Image(url='https://github.com/eric-haibin-lin/verl-community/blob/main/docs/multi_turn.png?raw=true', width=600)

$$
r_t=\exp(\log\frac{\pi_\theta}{\pi_{\text{old}}})
$$
- likelihood ratio (pg loss)
    - https://github.com/volcengine/verl/blob/main/verl/trainer/ppo/core_algos.py#L785-L788
- kl (kl_loss): k3 估计，也涉及 ratio ($\pi_{ref}$)
    - http://joschu.net/blog/kl-approx.html
    - https://github.com/volcengine/verl/blob/main/verl/trainer/ppo/core_algos.py#L1347C9-L1350C30

- https://verl.readthedocs.io/en/latest/advance/agent_loop.html#chat-completion-vs-token-in-token-out
    - Almost all agent frameworks (LangGraph, CrewAI, LlamaIndex, etc) call LLM with OpenAI chat completion api, and **keep chat history as messages**. So user may expect that we should use the chat completion api in multi-turn rollout.
    - But based on our recent experience on single-turn training on DAPO and multi-turn training on retool, we found the token_ids from apply the final messages may not equal to the token_ids by concat prompt_ids and response_ids in each turn.
        - encode(messages) != (prompt_ids ⊕ respose_ids)
    - This inconsistency is not a big problem for serving/agent system, but is **critical to RL training**. It causes the trajectory deviate from the policy model distribution. We have observed that `apply_chat_template` to the final chat history messages make PPO training not even converged in single-turn.
- 所以verl采用了token in and token out的方式，让agent调用llm generate方法时，输入和输出都使用token，来避免 token 和明文消息互相转换不一致的问题。
    - https://www.notion.so/verl-reTool-recipe-2398b5b7feba80a58156fa936f9f8de6
    - https://github.com/0russwest0/Agent-R1/issues/30#issuecomment-2826155367